In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd
import numpy as np
import polars as pl
import json
import time

from snapml import GraphFeaturePreprocessor

In [3]:
df = pl.read_csv("/content/drive/MyDrive/eth_tx_last4days_clean.csv")

df = df.sort("timestamp_unix")

all_addresses = pl.concat([df["from_address"], df["to_address"]]).unique()

address_mapping = pl.DataFrame({
    "address": all_addresses,
    "vertex_id": np.arange(0, len(all_addresses), dtype=np.int64)
})

df = df.join(address_mapping, left_on="from_address", right_on="address", how="left") \
       .rename({"vertex_id": "source_id"})

df = df.join(address_mapping, left_on="to_address", right_on="address", how="left") \
       .rename({"vertex_id": "target_id"})

df = df.with_columns(
    pl.Series(name="edge_id", values=np.arange(0, len(df), dtype=np.int64))
)

input_df = df.select([
    pl.col("edge_id").cast(pl.Float64),
    pl.col("source_id").cast(pl.Float64),
    pl.col("target_id").cast(pl.Float64),
    pl.col("timestamp_unix").cast(pl.Float64),
    pl.col("value_eth").cast(pl.Float64)
])

X_graph = input_df.to_numpy()

print(X_graph.shape)

(4290480, 5)


In [4]:
gp = GraphFeaturePreprocessor()

gp.set_params({
    "num_threads": 8,
    "time_window": 3600,
    "temp-cycle": True,
    "temp-cycle_tw": 3600,
    "temp-cycle_bins": [2, 3, 4, 5],
    "lc-cycle": True,
    "lc-cycle_tw": 3600,
    "lc-cycle_len": 5,
    "lc-cycle_bins": [2, 3, 4, 5],
    "vertex_stats": True,
    "vertex_stats_tw": 3600,
    "vertex_stats_cols": [4],
    "vertex_stats_feats": [0, 1, 4]
})



In [5]:
batch_size = 100000
total_rows = X_graph.shape[0]


chunks = []



In [6]:
for i in range(0, total_rows, batch_size):

    X_batch = X_graph[i : i + batch_size]

    X_batch_out = gp.transform(X_batch)

    chunks.append(X_batch_out)

    if i % 500000 == 0 or i + batch_size >= total_rows:
        print(f"{min(i + batch_size, total_rows)} / {total_rows}")

X_out = np.vstack(chunks)
print(X_out.shape)

100000 / 4290480
600000 / 4290480
1100000 / 4290480
1600000 / 4290480
2100000 / 4290480
2600000 / 4290480
3100000 / 4290480
3600000 / 4290480
4100000 / 4290480
4290480 / 4290480
(4290480, 83)


In [7]:
num_new_features = X_out.shape[1] - 5
snap_feature_names = [f"snap_motif_{i}" for i in range(num_new_features)]

all_columns = ["edge_id", "source_id", "target_id", "timestamp_unix", "value_eth"] + snap_feature_names

df_snap_edges = pl.DataFrame(X_out, schema=all_columns)

output_path = "/content/drive/MyDrive/snap_ml_edges_output.csv"
df_snap_edges.write_csv(output_path)

print(f"{output_path}")
print(df_snap_edges.shape)

/content/drive/MyDrive/snap_ml_edges_output.csv
(4290480, 83)


In [5]:
df_snap_edges = pl.read_csv("/content/drive/MyDrive/snap_ml_edges_output.csv")

snap_feature_names = df_snap_edges.columns[5:]

source_nodes = df_snap_edges.group_by("source_id").agg([
    pl.col(col).mean().alias(f"{col}_as_sender") for col in snap_feature_names
]).rename({"source_id": "vertex_id"})

target_nodes = df_snap_edges.group_by("target_id").agg([
    pl.col(col).mean().alias(f"{col}_as_receiver") for col in snap_feature_names
]).rename({"target_id": "vertex_id"})

df_snap_nodes = source_nodes.join(target_nodes, on="vertex_id", how="outer")

df_snap_nodes = df_snap_nodes.fill_null(0)

nodes_output_path = "/content/drive/MyDrive/snap_ml_nodes_output.csv"
df_snap_nodes.write_csv(nodes_output_path)

print(nodes_output_path)
print(df_snap_nodes.shape)

/tmp/ipykernel_176009/3782258289.py:15: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  df_snap_nodes = source_nodes.join(target_nodes, on="vertex_id", how="outer")


/content/drive/MyDrive/snap_ml_nodes_output.csv
(1397958, 158)
